# 4. Improving a model


Our,
Firt predictions can be called as baseline predictions

First model can be called as baseline model.

When dealing with model improvement, we can have 2 perspectives $\rightarrow$

*Form a data perspective:*

* Could we collect more data? 
* Could we improve our data?

*From a model perspective:*

* Is there a better model we could use?
* Could we improve the current model (hyper-parameter tuning)?

*Three ways to adjust hyper-parameters:*

* By Hand (train $\rightarrow$ validation $\rightarrow$ test)
* Randomly with RandomSearchCV
* Exhaustively with GridSearchCV

Here's an example to figure out what hyper-parameters a model is using:

In [1]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier()
clf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

To understand more about this, [doc](https://scikit-learn.org/stable/modules/grid_search.html) is the place to go.

# Adjust Hyper-Parameters by Hand


We'll perform this method on a classification model. From [doc](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#:~:text=datasets%20(n_samples%20%3E%3D%2010_000).-,Notes,-The%20default%20values) We're going ot try and adjust:

* `max_depth`
* `max_features`
* `min_samples_leaf`
* `min_samples_split`
* `n_estimators`

Let's make 3 sets: *Training*, *Validation*  & *Test*

![train-val-test](./img/train-val-test.png)
*[source](https://youtu.be/915Ao3zGNwA?si=1lvCNTtz9NRCLrzO&t=280)*

Making a handy-dandy function to take care of the evaluation metrics so that, we don't have to write them again and again:

In [2]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score

def evalute_prediction(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    metric_dict = {
        "accuracy": round(accuracy, 2),
        "precision": round(precision, 2),
        "f1_score": round(f1, 2),
        "recall": round(recall, 2),
    }
    print(f"Accuracy: {accuracy * 100 :.2f}%")
    print(f"Precision: {precision * 100 :.2f}%")
    print(f"Recall: {recall * 100 :.2f}%")
    print(f"F1_score: {f1 * 100 :.2f}%")
    
    return metric_dict

In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

In [4]:
np.random.seed(42)

df = pd.read_csv("./data/heart-disease.csv")

# suffle 100% of the data
df = df.sample(frac=1)

# Split into X & y
X = df.drop("target", axis=1)
y = df["target"]

# split the data into train, validation & test
# We'll do -> train(70%), validation(15%) & test (rest 15%)

# set boundary indexes
train_index = round(0.70 * df.shape[0])
val_index = train_index + round(0.15 * df.shape[0])

# Doing the split
X_train, y_train = X[:train_index], y[:train_index]
X_val, y_val = X[train_index:val_index], y[train_index:val_index]
X_test, y_test = X[val_index:], y[val_index:]

# checking if the splits covered whole X and y correcly
assert X_train.shape[0] + X_val.shape[0] + X_test.shape[0] == X.shape[0], "X split wasn't correct"
assert y_train.shape[0] + y_val.shape[0] + y_test.shape[0] == y.shape[0], "y split wasn't correct"

## Trial-0
Now let's build a model

In [5]:
clf = RandomForestClassifier(random_state=1)
clf.fit(X_train, y_train)

# Make baseline predictions using validation data
y_pred = clf.predict(X_val)

# Evaluate the classifier using validation data
base_line_metrics = evalute_prediction(y_val, y_pred)

Accuracy: 82.22%
Precision: 81.48%
Recall: 88.00%
F1_score: 84.62%


Let's try to improve the scores!

## Trial 1

We will create another model with different hyper parameter.

In this case let's try a different value for `n_estimator` $ = 40 $ and see the changes.

In [6]:
clf2 = RandomForestClassifier(n_estimators=40, random_state=1)
clf2.fit(X_train, y_train)

# use our validation data set to get the predictions
y_pred = clf2.predict(X_val)

# Evalute the new model with validation data set
base_line_metrics_2 = evalute_prediction(y_val, y_pred)

Accuracy: 84.44%
Precision: 84.62%
Recall: 88.00%
F1_score: 86.27%


Okay, so a bit imporvement. Let's keep moving.

## Trial-2

By reading the doc, This time we will add `max_depth` $=10$ to the estimator.

In [7]:
clf3 = RandomForestClassifier(n_estimators=40, max_depth=10, random_state=1)
clf3.fit(X_train, y_train)

y_pred = clf3.predict(X_val)

base_line_metrics_3 = evalute_prediction(y_val, y_pred)

Accuracy: 84.44%
Precision: 84.62%
Recall: 88.00%
F1_score: 86.27%


Okay, so we get the point. Play with different paramters to achieve some good numbers. In order to save time, let's just stop here.
And select `clf3` as our model to test on test data set and see the resutls.

## Final Test

In [8]:
y_pred = clf3.predict(X_test)

final_metrics = evalute_prediction(y_test, y_pred)

Accuracy: 86.96%
Precision: 88.00%
Recall: 88.00%
F1_score: 88.00%


Interpretation
---

Our model can correctly predict heart disease or not with an accuracy $\approx$ $87\%$. If our model predicts a heart disease case, it has a probability of $88\%$ of being correct. The model can correctly identify/recall $88\%$ of the acutal heart disease cases from the test data set. The F1_score is also $88\%$ suggesting that there is a good balance between precision and recall and it is closer to a $100\%$.

However, As we can already see, improving a model by hand manually can be a cumbersum process. Luckily, scikit learn has provided some good algorithms to overcome this challange. We will now look on RandomSearchCV or Randomize Search Cross Validation.

# Model Improvement: Random Search CV

In [11]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split


grid = {
    "n_estimators": [10, 100, 200, 500, 1000, 1200],
    "max_depth": [None, 5, 10, 20, 30],
    "max_features": ["auto", "sqrt"],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2, 4]
}

np.random.seed(42)

# Extract X, y
X = df.drop("target", axis=1)
y = df["target"]

# Split into train test, becasue RandomSearchCV will take care for validation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)


# Instansiate Classifier
clf = RandomForestClassifier(n_jobs=1)

# Setup RandomizedSearchCV
rs_clf = RandomizedSearchCV(
            estimator=clf,
            param_distributions=grid,
            n_iter=10,
            cv=5,
            verbose=2
        )

# Fit the RandomizedSearchCV version of clf
rs_clf.fit(X_train, y_train)



Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=2, min_samples_split=6, n_estimators=1200; total time=   1.6s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=2, min_samples_split=6, n_estimators=1200; total time=   1.8s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=2, min_samples_split=6, n_estimators=1200; total time=   1.7s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=2, min_samples_split=6, n_estimators=1200; total time=   1.7s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=2, min_samples_split=6, n_estimators=1200; total time=   1.7s
[CV] END max_depth=30, max_features=auto, min_samples_leaf=2, min_samples_split=4, n_estimators=100; total time=   0.2s


/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(
/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


[CV] END max_depth=30, max_features=auto, min_samples_leaf=2, min_samples_split=4, n_estimators=100; total time=   0.1s
[CV] END max_depth=30, max_features=auto, min_samples_leaf=2, min_samples_split=4, n_estimators=100; total time=   0.1s


/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(
/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


[CV] END max_depth=30, max_features=auto, min_samples_leaf=2, min_samples_split=4, n_estimators=100; total time=   0.2s
[CV] END max_depth=30, max_features=auto, min_samples_leaf=2, min_samples_split=4, n_estimators=100; total time=   0.1s


...

/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


[CV] END max_depth=10, max_features=auto, min_samples_leaf=4, min_samples_split=4, n_estimators=200; total time=   0.3s


/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


[CV] END max_depth=10, max_features=auto, min_samples_leaf=4, min_samples_split=4, n_estimators=200; total time=   0.3s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=2, min_samples_split=4, n_estimators=1000; total time=   1.4s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=2, min_samples_split=4, n_estimators=1000; total time=   1.5s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=2, min_samples_split=4, n_estimators=1000; total time=   1.3s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=2, min_samples_split=4, n_estimators=1000; total time=   1.4s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=2, min_samples_split=4, n_estimators=1000; total time=   1.4s


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(n_jobs=1),
                   param_distributions={'max_depth': [None, 5, 10, 20, 30],
                                        'max_features': ['auto', 'sqrt'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 4, 6],
                                        'n_estimators': [10, 100, 200, 500,
                                                         1000, 1200]},
                   verbose=2)

So, it has tried all the combinations from `grid` dictionary. And by deafault it stores the best combination of parameters.

In [12]:
rs_clf.best_params_

{'n_estimators': 200,
 'min_samples_split': 6,
 'min_samples_leaf': 2,
 'max_features': 'sqrt',
 'max_depth': None}

In [13]:
# Make predictions with the best hyper parameters
y_pred_rs = rs_clf.predict(X_test)

# Evaluate the predictions
rs_metrics = evalute_prediction(y_test, y_pred_rs)

Accuracy: 81.97%
Precision: 77.42%
Recall: 85.71%
F1_score: 81.36%


So, we have automated our `Model Improvement by Hand` approach.

## 3. GridSearchCV | Model Improvement

The key difference with`RandomizedSearchCV` is, instead of working with a definite number of approaches by `n_iter`, it uses a brutforce searching technique. So, it go through every single combinations provided in the `grid` variable

Now, what we can already guess is, GridSearchCV is computationally expensive than RandomizedSearchCV.

To mitigate this, we have couple of options:

1. shorten the lists in each keys of `grid` variable manually.
2. If already gone through `RandomizedSearchCV` step, use the `best_params_` dictionary smartly to focous around these points.

Let's do the second option.

In [17]:
# revisit best params
rs_clf.best_params_

{'n_estimators': 200,
 'min_samples_split': 6,
 'min_samples_leaf': 2,
 'max_features': 'sqrt',
 'max_depth': None}

In [18]:
# revist the grid
grid

{'n_estimators': [10, 100, 200, 500, 1000, 1200],
 'max_depth': [None, 5, 10, 20, 30],
 'max_features': ['auto', 'sqrt'],
 'min_samples_split': [2, 4, 6],
 'min_samples_leaf': [1, 2, 4]}

In [19]:
# Make another grid fouced on best_params
# so, we can make something like:

grid_2 = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None],
    'max_features': ['auto', 'sqrt'],
    'min_samples_split': [6],
    'min_samples_leaf': [1, 2]
}

Now let's see that in action

In [20]:
from sklearn.model_selection import GridSearchCV, train_test_split

np.random.seed(42)


# Extract X, y
X = df.drop("target", axis=1)
y = df["target"]

# Split into train test, becasue RandomSearchCV will take care for validation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)


# Instansiate Classifier
clf = RandomForestClassifier(n_jobs=1)

# Setup GridSearchCV
gs_clf = GridSearchCV(
            estimator=clf,
            param_grid=grid_2,
            cv=5,
            verbose=2
        )

# Fit the RandomizedSearchCV version of clf
gs_clf.fit(X_train, y_train)


Fitting 5 folds for each of 12 candidates, totalling 60 fits


/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


[CV] END max_depth=None, max_features=auto, min_samples_leaf=1, min_samples_split=6, n_estimators=100; total time=   0.3s


/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


[CV] END max_depth=None, max_features=auto, min_samples_leaf=1, min_samples_split=6, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=auto, min_samples_leaf=1, min_samples_split=6, n_estimators=100; total time=   0.2s


...

/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


[CV] END max_depth=None, max_features=auto, min_samples_leaf=2, min_samples_split=6, n_estimators=500; total time=   0.8s


/home/shifat/venvs/venvDS/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


[CV] END max_depth=None, max_features=auto, min_samples_leaf=2, min_samples_split=6, n_estimators=500; total time=   0.8s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=200; total time=   0.4s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=200; total time=   0.4s
[CV] END max_depth=None,

GridSearchCV(cv=5, estimator=RandomForestClassifier(n_jobs=1),
             param_grid={'max_depth': [None], 'max_features': ['auto', 'sqrt'],
                         'min_samples_leaf': [1, 2], 'min_samples_split': [6],
                         'n_estimators': [100, 200, 500]},
             verbose=2)

In [21]:
gs_clf.best_params_

{'max_depth': None,
 'max_features': 'sqrt',
 'min_samples_leaf': 1,
 'min_samples_split': 6,
 'n_estimators': 200}

In [22]:
y_pred_gs = gs_clf.predict(X_test)

gs_evaluation_metrics = evalute_prediction(y_test, y_pred_gs)

Accuracy: 78.69%
Precision: 74.19%
Recall: 82.14%
F1_score: 77.97%


In conclusion, It is safe to say that, We would actually use all three methods explained above, to do the hyper parameter tuning of our model.